# 04 YOLOv5 的三个检测尺度：为什么需要不同大小的特征图

你已理解 YOLOv1 的每格两个候选框。这一课转到当前项目使用的 YOLOv5，先回答一个新问题：**同一张图片中，远处小人脸和近处大人脸，为什么要结合不同分辨率的特征做预测？**

先区分两件事：检测尺度有几个，与每个位置有几个候选预测，不是一回事。

## 1. 小人脸在反复下采样后会怎样

你学 CNN 时知道，下采样会减小特征图的空间尺寸。随着网络深入，特征能够整合更大范围的信息，但空间表示也变得更粗。

设输入图中一张小人脸宽 16 像素：

- 在步长为 8 的特征网格上，它的宽度相当于约 2 个网格间距。
- 在步长为 32 的特征网格上，它的宽度只相当于约半个网格间距。

这不是说目标被精确压缩成几个像素，更不表示不足一格就绝对无法检测，而是说明：小目标在较粗的空间网格上更难被精细表示与定位。

因此，检测器希望同时利用分辨率较高的空间信息和经过深层处理的语义信息。

## 2. 一个 640 × 640 输入的具体例子

标准三尺度 YOLOv5 会在以下三个尺度上产生检测预测：

| 层级 | 相对于输入的步长 | 特征网格尺寸 | 通常对哪类目标更有帮助 |
|---|---:|---:|---|
| P3 | 8 | 80 × 80 | 小目标 |
| P4 | 16 | 40 × 40 | 中等目标 |
| P5 | 32 | 20 × 20 | 大目标 |

尺寸关系为：

$$
640/8=80,\qquad640/16=40,\qquad640/32=20
$$

这里固定使用方形 640 × 640 输入方便理解。当前网页的预处理使用保比例缩放和必要补边，实际输入可能是长方形，所以不保证每次都是这三个方形尺寸。

P3 的 3 表示该层级对应 $2^3=8$ 的下采样倍数；不是三个类别，也不是每处三个框。

## 3. 看同一目标在不同网格上的相对大小

![同一目标对应三个检测网格](assets/04_three_scales.png)

红框在三幅图中都表示输入图上的同一个 32 × 32 像素目标。横纵轴也都使用输入图坐标，变化的只是检测网格的间距。

| 网格步长 | 32 像素目标的宽度约跨几个间距 |
|---:|---:|
| 8 | 4 |
| 16 | 2 |
| 32 | 1 |

这是网格间距的示意图，不是真正的特征激活图。

**步长为 8，不代表每个特征点只看原图的 8 × 8 区域。** 特征点的感受野通常覆盖更大范围，还可能经过多尺度特征融合。

## 4. 不是把原图切成三套小图片

一次前向传播中，整张输入图片经过骨干网络产生不同层级的特征，再由 Neck 进行上采样、拼接和特征融合，检测部分利用三个尺度的特征分别预测。

高分辨率分支不仅使用浅层纹理，也会融合更深层的信息，因此不能把三个分支理解成互不交流的独立网络。

最终把各尺度的候选检测汇总，再进行分数筛选和 NMS 等后处理。

三个尺度让模型有机会兼顾大小不同的目标，不意味着某个目标必须由三个尺度各输出一个最终框。

## 5. “小、中、大”是倾向，不是硬性分界

P3 通常更适合小目标，P5 通常更适合大目标，但并不存在“某宽度以内永远只能由 P3 处理”的通用硬规则。

实际训练分配还会结合目标宽高与 anchor 的匹配等条件。一个真实目标也可能获得多个正样本监督，不能套用 YOLOv1 只在同格两框中选当前最大 IoU 那一套规则。

同一目标可能产生来自不同位置或尺度的重复预测，这也是后续需要筛选和 NMS 的原因之一。

本课只建立三尺度的直观理解，下一课再展开 anchor 与每个位置的输出。

## 6. 回到本地权重：三个尺度与三个 anchor

本次只读取了权重压缩包内 pickle 指令中的元数据，没有执行反序列化，也没有加载第二份模型到显存。可以核对到：

| 字段 | 数值 | 含义 |
|---|---:|---|
| nc | 3 | 当前口罩类别数 |
| nl | 3 | 检测尺度数量 |
| na | 3 | 每个尺度、每个位置的 anchor 数量 |
| no | 8 | 每个 anchor 对应预测的输出数，即 5 + 3 |

所以：**不是整张图片只有三个框，也不是每个位置只有一个框。** 三个尺度中，每个位置各有三个 anchor 对应的预测。

先把 anchor 理解为不同宽高的参考模板，预测会基于它调整出实际框；它不是已经确定的最终检测框。下一课会具体讲模板如何帮助预测。

## 7. 和你刚学过的 YOLOv1 对照

| 对比项 | 原论文 YOLOv1 的 VOC 设置 | 当前三尺度口罩 YOLOv5 |
|---|---|---|
| 组织预测的尺度 | 单个 7 × 7 网格 | 步长 8、16、32 三个尺度 |
| 每位置候选方式 | 两个直接预测的框 | 三个 anchor 对应的预测 |
| 类别输出 | 一个网格共享一组类别预测 | 每个 anchor 对应预测有自己的类别输出 |
| 训练责任分配 | 同格候选中按当前 IoU 选负责框 | 使用其自身的 anchor 匹配及邻近网格分配等规则 |

这里比较的是两个具体结构，不是所有称为 YOLO 的模型都固定使用这些数量。

学习 YOLOv1 帮助你理解“预测、标注、责任与损失”；正式学习 YOLOv5 时，需要重新对照它的具体结构和分配规则。

## 8. 自检

1. 640 × 640 输入，在步长 16 的尺度上，网格大小是多少？
2. P3 分辨率比 P5 高，这对远处的小人脸有什么帮助？
3. 步长为 8，是否表示每个特征点只看输入图的 8 × 8 像素？
4. 三个检测尺度，是否表示每张图片只产生三个候选框？

<details>
<summary>参考答案</summary>

1. 40 × 40。
2. 小人脸在较细的空间网格上通常保留更多可用于区分和定位的信息，但不保证一定检测成功。
3. 不是。步长描述空间网格相对于输入的间距，感受野描述一个特征点能够利用的输入范围。
4. 不是。每个尺度有许多位置，每个位置还对应多个 anchor 预测。

</details>

下一课：anchor 到底是什么，以及三类口罩模型每个候选预测为什么输出八个数。

## 本课依据与边界

- 本地 `models/mask_yolov5.pt`：静态读取元数据中的 nc、nl、na、no。
- 本地 `vendor/yolov5/models/yolo.py`：Detect 的层数、anchor 数和每个预测的输出数，以及各尺度输出的整理。
- 本地 `vendor/yolov5/models/yolov5s.yaml`：标准三尺度配置示例，Detect(P3,P4,P5)。这份当前源码配置不等于作者历史权重的每个内部模块都与其完全相同。
- 本地 `mask_app/detector.py`：网页实际的 letterbox 预处理。
- [Ultralytics 官方 YOLOv5 架构说明](https://docs.ultralytics.com/yolov5/tutorials/architecture-description)：模型结构、多尺度预测层与目标分配。

本地源码固定版本为 `af08ba156fd58ed5b0fe7e4e58a22c72c3cdd0ce`。本课未修改模型、启动训练或变更部署服务。